In [6]:
import ray
import requests
import time
from fastapi import FastAPI,Query
from ray import serve
from ml_parallel_pipeline import RollingOLSRegressionParallel
from symbols_list_distribution import symbolsList
import pandas as pd


app = FastAPI()

@serve.deployment
@serve.ingress(app)
class TrainAPI:
    @app.get("/")
    def get_portfolio_returns(self,index:str=Query(...),start_date:str=Query(...),end_date:str=Query(...),batch_size:int=Query(...)):
        symbols_list=symbolsList(index)
        model=RollingOLSRegressionParallel(symbols_list,start_date,end_date)
        portfolio_returns=model.train_parallel_pipeline(batch_size)
        return {
            'status':'success',
            # 'data':portfolio_returns.to_dict(orient='records'),
            'rows':portfolio_returns.shape[0],
            'columns':portfolio_returns.shape[1]
        }
    # @app.get('/')
    # def say_hello(self):
    #     return 'Hello world'

serve.start(deatached=True, http_options={'host':'0.0.0.0','port':8000})
serve.run(TrainAPI.bind(), route_prefix='/hello')
# resp = requests.get("http://localhost:8000/hello",params={'index':'downjones','start_date':'2015-09-29','end_date':'2023-09-27','batch_size':2})

# data=resp.json()

# df=pd.DataFrame(data['data'])

# df

    

INFO 2025-06-12 03:53:05,475 serve 851 -- Connecting to existing Serve app in namespace "serve". New http options will not be applied.
INFO 2025-06-12 03:53:05,524 serve 851 -- Connecting to existing Serve app in namespace "serve". New http options will not be applied.
WARNING 2025-06-12 03:53:05,526 serve 851 -- The new client HTTP config differs from the existing one in the following fields: ['host', 'location']. The new HTTP config is ignored.
(ServeController pid=1151) INFO 2025-06-12 03:53:05,616 controller 1151 -- Deploying new version of Deployment(name='TrainAPI', app='default') (initial target replicas: 1).
(ServeController pid=1151) INFO 2025-06-12 03:53:05,791 controller 1151 -- Stopping 1 replicas of Deployment(name='TrainAPI', app='default') with outdated versions.
(ServeController pid=1151) INFO 2025-06-12 03:53:05,792 controller 1151 -- Adding 1 replica to Deployment(name='TrainAPI', app='default').
(ServeController pid=1151) INFO 2025-06-12 03:53:08,256 controller 1151 

DeploymentHandle(deployment='TrainAPI')

[                       0%                       ]
[**********            20%                       ]  3 of 15 completed
[****************      33%                       ]  5 of 15 completed
[*************         27%                       ]  4 of 15 completed
[****************      33%                       ]  5 of 15 completed
[**********************47%                       ]  7 of 15 completed
[*******************   40%                       ]  6 of 15 completed
[**********************53%                       ]  8 of 15 completed
[**********************53%                       ]  8 of 15 completed
[*********************100%***********************]  15 of 15 completed
[*********************100%***********************]  15 of 15 completed
[                       0%                       ] [repeated 2x across cluster]
[**********************67%*******                ]  10 of 15 completed [repeated 5x across cluster]
[******                13%                       ]  2 of 15 complet

In [ ]:
!curl -sL "http://localhost:8000/hello/?index=downjones&start_date=2015-09-29&end_date=2023-09-27&batch_size=2"
